<a href="https://colab.research.google.com/github/madhuresh365/assignment-2-react-agent/blob/main/Copy_of_agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [2]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorboard 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam-v1 0.14.6 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-dataproc 5

In [4]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
DOCUMENTS = {
    "sample_protocol": (
        "Students must fill in documents in this section. "
        "Documents can be related to any topic - sports, manga, food, music etc. "
        "Have a minimum of 4 sentences with each document."
    ),
    "space_exploration": (
        "Space exploration has always captured human imagination through scientific discovery. "
        "The Apollo missions successfully landed the first humans on the Moon in 1969. "
        "Modern rovers like Perseverance continue to search for signs of ancient life on Mars. "
        "In the future, private space travel may make commercial orbital tourism a reality."
    )
}

In [5]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("minimum of 4"))


Students must fill in documents in this section. Documents can be related to any topic - sports, manga, food, music etc. Have a minimum of 4 sentences with each document.


Space exploration has always captured human imagination through scientific discovery. The Apollo missions successfully landed the first humans on the Moon in 1969. Modern rovers like Perseverance continue to search for signs of ancient life on Mars. In the future, private space travel may make commercial orbital tourism a reality.


In [14]:
# @title Ask gemini to search for something
!pip install google-genai --quiet

from google import genai
from google.genai import types
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="What is the sample protocol rule for students?",
    config=types.GenerateContentConfig(
        tools=[search_documents]
    )
)

print(response.text)

Based on the provided documents, the protocol rule for students regarding this section is:

* **Students must fill in documents in this section.** 
* The documents can be related to any topic (such as sports, manga, food, music, etc.).
* Each document must have a **minimum of 4 sentences**.


In [17]:
import time

client = genai.Client()  # reads GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find most the relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {
                "query": {"type": "string"},
            },
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)

model_name = "gemini-3.5-flash-lite"  # or gemini-3.5-flash-lite if traffic is high

def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=model_name,
                contents=contents,
                config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f" Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise

def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")

            # FIXED: calling search_documents instead of search_recipes
            result = search_documents(**args)

            print(f" Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )

        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"

_ = run_agent("What topics are students supposed to share documents on?")

Question: What topics are students supposed to share documents on?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'topics share documents'})
 Result preview: 
Students must fill in documents in this section. Documents can be related to any topic - sports, ma...

Final answer:
Students are supposed to share documents related to **any topic**—such as sports, manga, food, music, etc. Each document must have a minimum of 4 sentences.


In [18]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "What are the rules for students adding documents?",
    "What did the Apollo missions accomplish?",
    "How many ghosts eat carrots everyday?",
]

for q in questions:
    print("=" * 60)
    run_agent(q)
    print("\n")

Question: What are the rules for students adding documents?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'students adding documents rules'})
 Result preview: 
Students must fill in documents in this section. Documents can be related to any topic - sports, ma...

Final answer:
Based on the guidelines, the rules for students adding documents are:

* Students must fill in documents in this section.
* Documents can be related to any topic (e.g., sports, manga, food, music, etc.).
* Each document must have a minimum of 4 sentences.


Question: What did the Apollo missions accomplish?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'Apollo missions accomplish'})
 Result preview: 
Space exploration has always captured human imagination through scientific discovery. The Apollo mi...

Final answer:
Based on the provided documents, the Apollo missions successfully landed the first humans on the

In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)